In [ ]:
# Step 1: Install the necessary libraries
# This command will run automatically in environments like Google Colab.
try:
    import google.colab
    print("Running in Google Colab. Installing libraries...")
    # Switched from googletrans to the more reliable deep-translator
    !pip install -q Flask pyngrok deep-translator gTTS
except ImportError:
    print("Not running in Colab. Make sure to install libraries manually:")
    print("pip install Flask pyngrok deep-translator gTTS")

from flask import Flask, render_template_string, request, jsonify, send_file
from pyngrok import ngrok  # Import the pyngrok library
from deep_translator import GoogleTranslator # Import the new translator library
from gtts import gTTS
import io
import os

# Step 2: Create the Flask web application
app = Flask(__name__)

# A fixed dictionary of languages for the dropdown menus
fixed_languages = {
    'English': 'en',
    'Hindi': 'hi',
    'Bengali': 'bn',
    'Tamil': 'ta',
    'Telugu': 'te',
    'Marathi': 'mr',
    'Gujarati': 'gu',
    'Kannada': 'kn',
    'Malayalam': 'ml',
    'Punjabi': 'pa',
    'Urdu': 'ur',
}

# Step 3: Create the HTML, CSS, and JavaScript for the user interface
# This is all contained within a single string for simplicity.
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Language Translator</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600&display=swap');
        body {
            font-family: 'Poppins', sans-serif;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: #fff;
        }
        .container {
            width: 90%;
            max-width: 900px;
            background: rgba(255, 255, 255, 0.1);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            padding: 2rem;
            box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37);
            border: 1px solid rgba(255, 255, 255, 0.18);
        }
        h1 {
            text-align: center;
            margin-bottom: 2rem;
            font-weight: 600;
        }
        .translator-grid {
            display: grid;
            grid-template-columns: 1fr;
            gap: 2rem;
        }
        @media (min-width: 768px) {
            .translator-grid {
                grid-template-columns: 1fr 1fr;
            }
        }
        .card {
            background: rgba(255, 255, 255, 0.15);
            border-radius: 15px;
            padding: 1.5rem;
            display: flex;
            flex-direction: column;
        }
        select, textarea {
            width: 100%;
            background: rgba(0, 0, 0, 0.2);
            border: 1px solid rgba(255, 255, 255, 0.3);
            border-radius: 8px;
            color: #fff;
            padding: 0.75rem;
            font-family: 'Poppins', sans-serif;
            font-size: 1rem;
            margin-bottom: 1rem;
        }
        select option {
            background: #3c3c3c;
        }
        textarea {
            flex-grow: 1;
            min-height: 150px;
            resize: vertical;
        }
        textarea::placeholder {
            color: rgba(255, 255, 255, 0.6);
        }
        .card-footer {
            display: flex;
            justify-content: flex-end;
            align-items: center;
            gap: 0.5rem;
            margin-top: 1rem;
        }
        .icon-button {
            background: none;
            border: none;
            cursor: pointer;
            color: #fff;
            opacity: 0.8;
            transition: opacity 0.2s;
        }
        .icon-button:hover {
            opacity: 1;
        }
        .translate-button-container {
            display: flex;
            justify-content: center;
            margin-top: 2rem;
        }
        #translate-btn {
            padding: 0.8rem 2.5rem;
            font-size: 1.1rem;
            font-weight: 600;
            background-color: #fff;
            color: #667eea;
            border: none;
            border-radius: 50px;
            cursor: pointer;
            transition: transform 0.2s, box-shadow 0.2s;
            box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        }
        #translate-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 6px 20px rgba(0,0,0,0.3);
        }
        #translate-btn:disabled {
            background-color: #ccc;
            cursor: not-allowed;
        }
        #error-message {
            text-align: center;
            color: #ffbaba;
            margin-top: 1rem;
            height: 1.2em;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Language Translator</h1>
        <div class="translator-grid">
            <!-- Input Card -->
            <div class="card">
                <select id="source-lang">
                    {% for name, code in languages.items() %}
                        <option value="{{ code }}">{{ name }}</option>
                    {% endfor %}
                </select>
                <textarea id="input-text" placeholder="Enter text to translate..."></textarea>
            </div>
            <!-- Output Card -->
            <div class="card">
                <select id="target-lang">
                     {% for name, code in languages.items() %}
                        <option value="{{ code }}" {% if code == 'hi' %}selected{% endif %}>{{ name }}</option>
                    {% endfor %}
                </select>
                <textarea id="output-text" placeholder="Translation will appear here..." readonly></textarea>
                <div class="card-footer">
                    <button id="copy-btn" class="icon-button" title="Copy to clipboard">
                        <svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2" ry="2"></rect><path d="M5 15H4a2 2 0 0 1-2-2V4a2 2 0 0 1 2-2h9a2 2 0 0 1 2 2v1"></path></svg>
                    </button>
                    <button id="listen-btn" class="icon-button" title="Listen to translation">
                        <svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><polygon points="11 5 6 9 2 9 2 15 6 15 11 19 11 5"></polygon><path d="M19.07 4.93a10 10 0 0 1 0 14.14M15.54 8.46a5 5 0 0 1 0 7.07"></path></svg>
                    </button>
                </div>
            </div>
        </div>
        <div class="translate-button-container">
            <button id="translate-btn">Translate</button>
        </div>
        <p id="error-message"></p>
    </div>

    <script>
        const translateBtn = document.getElementById('translate-btn');
        const inputText = document.getElementById('input-text');
        const outputText = document.getElementById('output-text');
        const sourceLang = document.getElementById('source-lang');
        const targetLang = document.getElementById('target-lang');
        const copyBtn = document.getElementById('copy-btn');
        const listenBtn = document.getElementById('listen-btn');
        const errorMessage = document.getElementById('error-message');

        translateBtn.addEventListener('click', async () => {
            const text = inputText.value;
            const src = sourceLang.value;
            const dest = targetLang.value;

            if (!text.trim()) return;

            translateBtn.disabled = true;
            translateBtn.textContent = 'Translating...';
            outputText.value = '';
            errorMessage.textContent = '';

            try {
                const response = await fetch('/translate', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ text, src, dest })
                });

                const data = await response.json();

                if (response.ok) {
                    outputText.value = data.translation;
                } else {
                    errorMessage.textContent = data.error || 'An unknown error occurred.';
                }
            } catch (error) {
                errorMessage.textContent = 'Failed to connect to the server.';
            } finally {
                translateBtn.disabled = false;
                translateBtn.textContent = 'Translate';
            }
        });

        copyBtn.addEventListener('click', () => {
            if(outputText.value) {
                navigator.clipboard.writeText(outputText.value)
                    .then(() => {
                        // Optional: Show a "Copied!" message
                        const originalIcon = copyBtn.innerHTML;
                        copyBtn.innerHTML = '<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"></polyline></svg>';
                        setTimeout(() => copyBtn.innerHTML = originalIcon, 2000);
                    })
                    .catch(err => console.error('Failed to copy: ', err));
            }
        });

        listenBtn.addEventListener('click', () => {
            const text = outputText.value;
            const lang = targetLang.value;
            if (!text.trim()) return;

            const audio = new Audio(`/speak?text=${encodeURIComponent(text)}&lang=${lang}`);
            audio.play().catch(e => {
                errorMessage.textContent = "Could not play audio. Please interact with the page first.";
            });
        });

    </script>
</body>
</html>
"""

# Step 4: Define the backend routes (API endpoints)

@app.route('/')
def home():
    """Renders the main HTML page."""
    return render_template_string(HTML_TEMPLATE, languages=fixed_languages)

@app.route('/translate', methods=['POST'])
def translate():
    """Handles the translation request from the frontend."""
    data = request.json
    text = data.get('text')
    src = data.get('src')
    dest = data.get('dest')

    if not text:
        return jsonify({'error': 'No text provided'}), 400

    try:
        # Use the new deep-translator library
        translation = GoogleTranslator(source=src, target=dest).translate(text)
        return jsonify({'translation': translation})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/speak')
def speak():
    """Handles the text-to-speech request and returns an audio file."""
    text = request.args.get('text')
    lang = request.args.get('lang')

    if not text:
        return "No text provided", 400

    try:
        tts = gTTS(text=text, lang=lang)
        audio_fp = io.BytesIO()
        tts.write_to_fp(audio_fp)
        audio_fp.seek(0)
        return send_file(audio_fp, mimetype='audio/mpeg')
    except Exception as e:
        return str(e), 500

# Step 5: Run the Flask application
if __name__ == '__main__':
    # --- ADD YOUR NGROK AUTHTOKEN HERE ---
    # Get your free token from https://dashboard.ngrok.com/get-started/your-authtoken
    NGROK_AUTH_TOKEN = "paste your token here only here paste"

    # Check if the token has been pasted
    if NGROK_AUTH_TOKEN == "PASTE_YOUR_NGROK_AUTHTOKEN_HERE" or NGROK_AUTH_TOKEN == "paste here token only here paste":
        print("ERROR: Please paste your ngrok authtoken into the script.")
    else:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)

        # Set the port for the Flask app
        port = 5000
        # Open a http tunnel on the specified port
        public_url = ngrok.connect(port)
        print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")
        # Start the Flask app
        app.run(port=port)


Running in Google Colab. Installing libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.2 MB/s eta 0:00:00
 * ngrok tunnel "NgrokTunnel: "https://21197b6ce1bb.ngrok-free.app" -> "http://localhost:5000"" -> "http://127.0.0.1:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Aug/2025 14:18:29] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Aug/2025 14:18:30] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [21/Aug/2025 14:18:42] "POST /translate HTTP/1.1" 200 -
